# 서울시 지하철·버스 CSV 스키마 탐색

본 노트북은 PROJECT_PLAN.md의 **[3] Spark 전처리** 단계 전에,
raw CSV의 **컬럼 구조·인코딩·이상치**를 먼저 확인하기 위한 탐색용 노트북이다.

## 실행 환경

| Part | 환경 | 목적 |
|---|---|---|
| **Part 1** | 로컬 PC + Jupyter + pandas | 빠른 스키마/인코딩 확인 |
| **Part 2** | Sandbox HDP + pyspark (또는 Zeppelin) | 실제 작업 환경에서 재현·검증 |

## 검증할 사항

1. 컬럼명 형식 — 한글 vs 영문
2. CSV 인코딩 — UTF-8 / CP949 중 어느 쪽?
3. 시간대 컬럼 구조 (wide format 형태 파악)
4. `subway_202603.csv` 이상치 원인 — 파일 크기가 다른 달의 약 2배

## 기 검증된 사실 (Sandbox pyspark 셸 사전 탐색 결과)

- 컬럼명이 **영문**으로 제공됨 (`USE_MM`, `SBWY_ROUT_LN_NM`, `STTN`, `HR_n_GET_ON_NOPE`, `HR_n_GET_OFF_NOPE`, `JOB_YMD`)
- `subway_202412.csv` 기준: 총 52개 컬럼 (메타 4 + 시간대 48), 621행
- 시간대 순서: 4시 ~ 3시 (서울 지하철 운영시간)
- CP949로 읽으면 데이터값(역명) 한글이 깨짐 → **UTF-8 가능성 확인 필요**
- Sandbox pyspark는 Python 2.7이라 `show()`에서 `UnicodeEncodeError` 발생 → stdout writer 교체로 해결


---

## Part 1: 로컬에서 pandas로 빠른 스키마·인코딩 확인

목적: PySpark 띄우는 비용 없이, raw CSV의 인코딩과 컬럼 구조를 즉시 확인.


In [ ]:
# 본인 로컬 경로로 수정
RAW_DIR = r"C:\\path\\to\\BP_clone\\src\\ingest\\data\\raw"

import os
SUBWAY_202412 = os.path.join(RAW_DIR, "subway_202412.csv")
SUBWAY_202603 = os.path.join(RAW_DIR, "subway_202603.csv")
BUS_202412    = os.path.join(RAW_DIR, "bus_202412.csv")

print("subway_202412 exists:", os.path.exists(SUBWAY_202412))
print("subway_202603 exists:", os.path.exists(SUBWAY_202603))
print("bus_202412    exists:", os.path.exists(BUS_202412))


### 1-1. 인코딩 결정 (UTF-8 vs CP949)

같은 파일을 두 인코딩으로 읽어 한글이 깨지지 않는 쪽을 정답으로 채택.


In [ ]:
import pandas as pd

for enc in ["utf-8", "cp949"]:
    try:
        df = pd.read_csv(SUBWAY_202412, encoding=enc, nrows=3)
        meta_cols = df.columns.tolist()[:3]
        print("--- encoding={} ---".format(enc))
        print("Columns (first 3):", meta_cols)
        print(df[meta_cols].head(2).to_string(index=False))
        print()
    except UnicodeDecodeError as e:
        print("encoding={} FAILED: {}".format(enc, e))
        print()


**정답 판정**: 호선명(`1호선`)·역명(`서울역`, `시청`)이 한글로 정상 표시되는 인코딩을 채택.
아래 셀의 `ENC` 변수를 결과에 맞게 수정.


In [ ]:
ENC = "utf-8"  # 위 결과에 맞게 변경 가능


### 1-2. Subway 스키마 전체 확인


In [ ]:
df_sub = pd.read_csv(SUBWAY_202412, encoding=ENC)
print("Subway shape:", df_sub.shape)
print("Subway columns ({}):".format(len(df_sub.columns)))
for c in df_sub.columns:
    print("  -", c)
df_sub.head(3)


### 1-3. Bus 스키마 전체 확인


In [ ]:
df_bus = pd.read_csv(BUS_202412, encoding=ENC)
print("Bus shape:", df_bus.shape)
print("Bus columns ({}):".format(len(df_bus.columns)))
for c in df_bus.columns:
    print("  -", c)
df_bus.head(3)


### 1-4. `subway_202603` 이상치 원인 파악

파일 크기가 `subway_202412`(~221KB)의 약 2배인 ~445KB로 측정됨.
가설: (a) 행 수 2배 (중복?), (b) 추가 컬럼, (c) 데이터 자체가 풍부해진 신규 역.


In [ ]:
df_603 = pd.read_csv(SUBWAY_202603, encoding=ENC)
print("202412 rows:", len(df_sub))
print("202603 rows:", len(df_603))
print("202603 columns:", len(df_603.columns))
print("Column diff vs 202412:",
      set(df_603.columns) ^ set(df_sub.columns) or "동일")
print()
print("202603 row count ratio: {:.2f}x".format(len(df_603) / len(df_sub)))


In [ ]:
# 중복 행 확인 - 메타 3개 키(USE_MM, SBWY_ROUT_LN_NM, STTN) 기준
key_cols = df_603.columns[:3].tolist()
dup = (df_603.groupby(key_cols).size()
              .reset_index(name="count")
              .query("count > 1")
              .sort_values("count", ascending=False))
print("중복 키 조합 수:", len(dup))
print(dup.head(10).to_string(index=False))


---

## Part 2: PySpark로 검증 (Sandbox HDP / Zeppelin)

HDFS에 이미 적재된 raw 데이터(`/user/maria_dev/seoul/raw/`)에 대해 동일한 확인을 수행.

**실행 위치 선택**:
- **Zeppelin**: 각 셀 첫 줄에 `%spark2.pyspark` 매직 추가 후 `Shift+Enter`
- **Sandbox SSH (pyspark 셸)**: 셀 1·2를 먼저 실행
- **로컬 PySpark**: HDFS 경로 대신 로컬 경로로 바꿔서 실행


In [ ]:
# 로컬에서 직접 PySpark 실행할 때만 사용 (Sandbox/Zeppelin은 spark가 이미 준비됨)
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.master("local[*]").appName("explore").getOrCreate()

HDFS_SUBWAY_202412 = "/user/maria_dev/seoul/raw/subway/subway_202412.csv"
HDFS_SUBWAY_202603 = "/user/maria_dev/seoul/raw/subway/subway_202603.csv"
HDFS_BUS_202412    = "/user/maria_dev/seoul/raw/bus/bus_202412.csv"

print("Spark version:", spark.version)


### 2-1. Python 2 stdout 인코딩 fix (Sandbox pyspark 셸에서만 필요)

Sandbox HDP의 pyspark 셸은 Python 2.7로 실행되며,
`show()` 출력 시 한글이 포함되면 `UnicodeEncodeError: 'ascii' codec...` 발생.
stdout writer를 UTF-8로 교체하여 해결.

> Zeppelin / Python 3 환경에서는 불필요 (try-except로 안전하게 처리).


In [ ]:
import sys
try:
    import codecs
    sys.stdout = codecs.getwriter("utf-8")(sys.stdout)
    print("stdout writer -> utf-8 OK")
except (TypeError, AttributeError):
    print("Python 3 / Zeppelin: no fix needed")


### 2-2. Subway 로드 (UTF-8 가정 — Part 1 결과 반영)


In [ ]:
df_sub_sp = (spark.read
             .option("header", "true")
             .option("encoding", "UTF-8")
             .csv(HDFS_SUBWAY_202412))

df_sub_sp.printSchema()
print("Columns count:", len(df_sub_sp.columns))
print("Row count:", df_sub_sp.count())

# 메타 컬럼만 5행 - 한글 깨짐 여부 확인
df_sub_sp.select("USE_MM", "SBWY_ROUT_LN_NM", "STTN").show(5, truncate=False)


### 2-3. Bus 로드


In [ ]:
df_bus_sp = (spark.read
             .option("header", "true")
             .option("encoding", "UTF-8")
             .csv(HDFS_BUS_202412))

df_bus_sp.printSchema()
print("Columns count:", len(df_bus_sp.columns))
print("Row count:", df_bus_sp.count())

# 메타 컬럼 6개 - 한글 깨짐 여부 확인
df_bus_sp.select(df_bus_sp.columns[:6]).show(3, truncate=False)


### 2-4. `subway_202603` 이상치 검증


In [ ]:
df_603_sp = (spark.read
             .option("header", "true")
             .option("encoding", "UTF-8")
             .csv(HDFS_SUBWAY_202603))

print("202412 rows:", df_sub_sp.count())
print("202603 rows:", df_603_sp.count())

# 메타 3개 키 기준 중복 확인
key_cols = df_603_sp.columns[:3]
df_603_sp.groupBy(*key_cols).count().filter("count > 1").show(10, truncate=False)


---

## 발견 사항 및 다음 단계

### A. 컬럼명 매핑 (영문 → 한글 의미)

| 의미 | Subway | Bus (예상) |
|---|---|---|
| 사용월 | `USE_MM` | (확인 필요) |
| 호선/노선 | `SBWY_ROUT_LN_NM` | (확인 필요) |
| 역/정류장 | `STTN` | (확인 필요) |
| 시간 n시 승차 | `HR_n_GET_ON_NOPE` | (확인 필요) |
| 시간 n시 하차 | `HR_n_GET_OFF_NOPE` | (확인 필요) |
| 작업일자 | `JOB_YMD` | (확인 필요) |

→ WORKFLOW.md `preprocess_subway.py`/`preprocess_bus.py`의
`withColumnRenamed("사용월", ...)` 부분을 영문 키 기준으로 수정해야 함.

### B. wide → long 변환 로직 (시간 추출)

WORKFLOW.md 가정 (한글):
```python
digits = [int(s) for s in c.split("시")[0].split() if s.isdigit()]
```

영문 컬럼에 맞춘 수정:
```python
# 예: "HR_4_GET_ON_NOPE" -> 4, kind="boarding"
parts = c.split("_")  # ["HR", "4", "GET", "ON", "NOPE"]
h = int(parts[1])
kind = "boarding" if "ON" in c else "alighting"
```

### C. 다음 단계

1. 이 노트북을 로컬에서 먼저 실행 → 인코딩(UTF-8/CP949) 확정 + Bus 컬럼 매핑 추출
2. Zeppelin에 동일 셀 복붙해서 HDFS 경로로 재검증
3. WORKFLOW.md [3-2]·[3-3] `preprocess_subway.py`, `preprocess_bus.py` 작성
   - 위 영문 매핑 적용
   - 확정 인코딩 적용
   - Parquet 저장 (`year=YYYY/month=MM` 파티션)
